# LoRA Fine-Tuning with PEFT on Kubeflow Trainer

This guide demonstrates how to fine-tune a language model using
[LoRA (Low-Rank Adaptation)](https://arxiv.org/abs/2106.09685) with the
[PEFT library](https://huggingface.co/docs/peft) on Kubeflow Trainer.

LoRA is a parameter-efficient fine-tuning technique that freezes the pretrained model weights
and injects trainable low-rank decomposition matrices into each layer of the Transformer
architecture. This dramatically reduces the number of trainable parameters and GPU memory
requirements.

**Why LoRA matters for infrastructure:**
- Reduces GPU memory usage by 60-80% compared to full fine-tuning
- Saves only adapter weights (~1-10 MB) instead of the full model (hundreds of MB to GB)
- Enables fine-tuning large models on smaller hardware
- Multiple task-specific adapters can share one base model

This example uses:
- **Model**: [distilgpt2](https://huggingface.co/distilgpt2) (82M parameters, no HuggingFace token required)
- **Dataset**: [Abirate/english_quotes](https://huggingface.co/datasets/Abirate/english_quotes) (~2500 quotes)
- **Technique**: LoRA with only ~0.3% trainable parameters (294K out of 82M)

Part of https://github.com/kubeflow/trainer/issues/2040

# Install the Kubeflow SDK

You need to install the Kubeflow SDK to interact with Kubeflow Trainer APIs:

In [ ]:
# !pip install -U kubeflow

Install dependencies for LoRA fine-tuning:

In [ ]:
!pip install "peft>=0.11" "transformers[torch]" "datasets"

# Understanding LoRA

LoRA (Low-Rank Adaptation) works by adding pairs of low-rank matrices to existing model weights
instead of modifying the original weights directly. Key configuration parameters:

- **Rank (`r`)**: The dimension of the low-rank matrices. Lower rank means fewer trainable parameters.
  Typical values: 4, 8, 16, 32. We use `r=8`.
- **Alpha (`lora_alpha`)**: Scaling factor for the LoRA update. Controls how much the adapter
  influences the output. Common practice is `alpha = 2 * r`. We use `lora_alpha=16`.
- **Dropout (`lora_dropout`)**: Dropout probability for LoRA layers, helps prevent overfitting.
  We use `lora_dropout=0.05`.
- **Target modules (`target_modules`)**: Which layers to apply LoRA to. For GPT-2 style models,
  the attention projection layers `c_attn` and `c_proj` are standard targets.

The key insight is that `get_peft_model()` freezes all base model weights (setting
`requires_grad=False`) and only the injected LoRA adapter parameters remain trainable.
When saving, only the tiny adapter weights are persisted (~1 MB), not the full model.

# Define the LoRA training function

We wrap the training script into a function to create the Kubeflow TrainJob.
All imports must be inside the function since it will be serialized and sent to worker nodes.

In [ ]:
def train_lora(model_name="distilgpt2", num_samples=500, lora_rank=8):
    import os

    import torch
    import torch.distributed as dist
    from datasets import load_dataset
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        DataCollatorForLanguageModeling,
        Trainer,
        TrainingArguments,
    )

    # Initialize distributed environment
    _, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    dist.init_process_group(backend=backend)

    local_rank = int(os.getenv("LOCAL_RANK", 0))
    print(
        "Distributed Training with WORLD_SIZE: {}, RANK: {}, LOCAL_RANK: {}.".format(
            dist.get_world_size(),
            dist.get_rank(),
            local_rank,
        )
    )

    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    # GPT-2 models do not have a pad token by default
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id

    # Configure LoRA
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=lora_rank,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["c_attn", "c_proj"],
    )

    # Apply LoRA adapters to the model
    # This freezes all base model weights and only adapter params are trainable
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    # Expected output: trainable params: 294,912 || all params: 82,190,592 || trainable%: 0.3588

    # Load and prepare the dataset
    dataset = load_dataset("Abirate/english_quotes", split=f"train[:{num_samples}]")

    def tokenize_function(examples):
        return tokenizer(
            examples["quote"],
            truncation=True,
            max_length=128,
            padding="max_length",
        )

    tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)
    split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)

    # Data collator for causal language modeling (no masking)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    # Define training arguments
    training_args = TrainingArguments(
        output_dir=f"lora-{model_name}",
        learning_rate=2e-4,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=10,
        push_to_hub=False,
        remove_unused_columns=False,
    )

    # Create the Trainer and start training
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=split_dataset["train"],
        eval_dataset=split_dataset["test"],
        data_collator=data_collator,
    )

    trainer.train()

    # Save the LoRA adapter only (not the full model)
    # On rank 0 only, to avoid conflicts in distributed training
    if dist.get_rank() == 0:
        adapter_path = f"lora-{model_name}/adapter"
        model.save_pretrained(adapter_path)
        print(f"LoRA adapter saved to {adapter_path}")
        print("Saved files are only the adapter weights (~1 MB), not the full model.")

# List available Training Runtimes

Check the available runtimes before submitting the TrainJob:

In [ ]:
from kubeflow.trainer import TrainerClient, CustomTrainer

for r in TrainerClient().list_runtimes():
    print(f"Name: {r.name}, Framework: {r.trainer.framework}, Trainer Type: {r.trainer.trainer_type.value}")

# Submit the LoRA TrainJob

Use `TrainerClient().train()` to submit the training function as a distributed TrainJob.
The `packages_to_install` parameter ensures the required libraries are installed on the worker nodes.

In [ ]:
MODEL_NAME = "distilgpt2"
args = {
    "model_name": MODEL_NAME,
    "num_samples": 500,
    "lora_rank": 8,
}

job_id = TrainerClient().train(
    trainer=CustomTrainer(
        func=train_lora,
        func_args=args,
        num_nodes=1,
        packages_to_install=["peft>=0.11", "datasets", "transformers[torch]"],
        resources_per_node={
            "cpu": "2",
            "memory": "12Gi",
            # Uncomment this to distribute the TrainJob using GPU nodes
            # "nvidia.com/gpu": 1,
        },
    ),
)

In [ ]:
# Train API generates a random TrainJob id.
job_id

# Check the TrainJob details

Use `list_jobs()` and `get_job()` APIs to get details about the created TrainJob and its steps.

In [ ]:
for job in TrainerClient().list_jobs():
    print(f"TrainJob: {job.name}, Status: {job.status}, Created at: {job.creation_timestamp}")

In [ ]:
# Wait for the running status.
TrainerClient().wait_for_job_status(name=job_id, status={"Running"})

In [ ]:
for c in TrainerClient().get_job(name=job_id).steps:
    print(f"Step: {c.name}, Status: {c.status}, Devices: {c.device} x {c.device_count}")

# Show the TrainJob logs

Use `get_job_logs()` API to retrieve the TrainJob logs.
Verify that the logs show approximately 0.3% trainable parameters and decreasing loss.

In [ ]:
for logline in TrainerClient().get_job_logs(job_id, follow=True):
    print(logline)

# Extension: QLoRA (Quantized LoRA)

[QLoRA](https://arxiv.org/abs/2305.14314) combines LoRA with 4-bit quantization to further
reduce memory usage. The base model is loaded in 4-bit precision using `bitsandbytes`, while
LoRA adapters are trained in full precision.

**Requirements**: QLoRA requires a CUDA GPU and the `bitsandbytes` library.
It is not compatible with CPU-only environments.

Below is an example of how to modify the training function for QLoRA.
This code is provided for reference only and is not executed in this notebook.

In [ ]:
# QLoRA example - requires GPU and bitsandbytes
# Uncomment and modify the training function to use QLoRA:

# from transformers import BitsAndBytesConfig
#
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )
#
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=quantization_config,
#     device_map={"": local_rank},
# )
#
# # Then apply LoRA as usual:
# model = get_peft_model(model, lora_config)

# Comparison: Full Fine-Tuning vs LoRA vs QLoRA

| Aspect | Full Fine-Tuning | LoRA | QLoRA |
|---|---|---|---|
| **Trainable Parameters** | 100% (82M) | ~0.36% (294K) | ~0.36% (294K) |
| **GPU Memory (distilgpt2)** | ~1.5 GB | ~0.5 GB | ~0.3 GB |
| **Saved Artifact Size** | ~330 MB | ~1 MB (adapter only) | ~1 MB (adapter only) |
| **Base Model Weights** | Modified | Frozen | Frozen + Quantized (4-bit) |
| **CPU Support** | Yes | Yes | No (requires CUDA) |
| **Extra Dependencies** | None | `peft` | `peft`, `bitsandbytes` |
| **Inference** | Direct | Merge adapter or load on-the-fly | Merge adapter or load on-the-fly |

# Clean up

To delete the TrainJob you can use the `delete_job()` API and pass the generated `job_id`.

In [ ]:
# _ = TrainerClient().delete_job(job_id)